# Train Audio Librosa Models

Run this notebook for the audio-only modeling task.

Inputs:
- `data/audio/master_audio_features.csv`
- `data/audio/billboard_top_100_all_years.csv` or `data/audio/top_100_billboard_songs/Billboard_Top_100_*.csv`

Outputs saved to Drive:
- `CLARIFY/audio/model_outputs/audio_recommender_index.joblib`
- `CLARIFY/audio/model_outputs/audio_hit_score_models.joblib`
- `CLARIFY/audio/model_outputs/audio_model_metadata.json`
- `CLARIFY/data/audio/billboard_hit_score_targets.csv`

The hit-score target is derived from Billboard year-end rank: `HitScore = (101 - Billboard_Rank) / 100`.

## 1. Setup

Installs/imports the basic data science libraries used in this notebook.

In [ ]:
!pip install -q pandas numpy scikit-learn joblib

In [ ]:
from pathlib import Path
import json
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

## 2. Paths

Finds the input CSV from the repo or Drive, then saves all model artifacts to `/content/drive/MyDrive/CLARIFY`.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/CLARIFY')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    DRIVE_ROOT,
    Path('/content/DS3-CLARIFY'),
    Path('/content/drive/MyDrive/DS3-CLARIFY'),
]
PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / 'data' / 'audio' / 'master_audio_features.csv').exists()),
    DRIVE_ROOT,
)

AUDIO_DATA_DIR = PROJECT_ROOT / 'data' / 'audio'
BILLBOARD_CHART_DIR = AUDIO_DATA_DIR / 'top_100_billboard_songs'
BILLBOARD_COMBINED_PATH = AUDIO_DATA_DIR / 'billboard_top_100_all_years.csv'
DRIVE_AUDIO_DATA_DIR = DRIVE_ROOT / 'data' / 'audio'
MODEL_DIR = DRIVE_ROOT / 'audio' / 'model_outputs'
DRIVE_AUDIO_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MASTER_AUDIO_PATH = AUDIO_DATA_DIR / 'master_audio_features.csv'
BILLBOARD_TARGETS_PATH = DRIVE_AUDIO_DATA_DIR / 'billboard_hit_score_targets.csv'
AUDIO_RECOMMENDER_PATH = MODEL_DIR / 'audio_recommender_index.joblib'
AUDIO_HIT_MODELS_PATH = MODEL_DIR / 'audio_hit_score_models.joblib'
AUDIO_METADATA_PATH = MODEL_DIR / 'audio_model_metadata.json'

TARGET_COLUMN = 'HitScore'
RANK_COLUMNS = ['Billboard_Rank', 'Rank', 'Peak_Rank', 'peak_rank']

print('Project root:', PROJECT_ROOT)
print('Drive artifact root:', DRIVE_ROOT)
print('Master audio:', MASTER_AUDIO_PATH, MASTER_AUDIO_PATH.exists())
print('Billboard combined file:', BILLBOARD_COMBINED_PATH, BILLBOARD_COMBINED_PATH.exists())
print('Billboard chart dir:', BILLBOARD_CHART_DIR, BILLBOARD_CHART_DIR.exists())
print('Audio artifacts save to:', MODEL_DIR)

## 3. Load Librosa Features

Loads the master audio table and verifies that tempo, MFCC, chroma, and spectral centroid columns exist.

In [ ]:
audio_df = pd.read_csv(MASTER_AUDIO_PATH)

IDENTITY_COLUMNS = ['SONG_ID', 'SONG_TITLE', 'ARTIST_NAME']
LIBROSA_FEATURE_COLUMNS = (
    ['tempo']
    + [f'mfcc_{i}' for i in range(1, 14)]
    + [f'chroma_mean_{i}' for i in range(1, 13)]
    + [f'chroma_std_{i}' for i in range(1, 13)]
    + ['spectral_centroid']
)
OPTIONAL_METADATA_FEATURES = ['year']

def normalize_text(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).lower())

def make_song_artist_key(df, title_col, artist_col):
    return df[title_col].map(normalize_text) + '|' + df[artist_col].map(normalize_text)

def load_billboard_targets(chart_dir, combined_file=None):
    if combined_file is not None and Path(combined_file).exists():
        targets = pd.read_csv(combined_file)
        rename_map = {'Song Name': 'SONG_TITLE', 'Artist Name': 'ARTIST_NAME'}
        targets = targets.rename(columns=rename_map)
        required = {'SONG_TITLE', 'ARTIST_NAME', 'Billboard_Year', 'Billboard_Rank'}
        missing = required - set(targets.columns)
        if missing:
            raise ValueError(f'{combined_file} missing columns: {sorted(missing)}')
        targets['Billboard_Year'] = pd.to_numeric(targets['Billboard_Year'], errors='coerce').astype('Int64')
        targets['Billboard_Rank'] = pd.to_numeric(targets['Billboard_Rank'], errors='coerce')
        if TARGET_COLUMN not in targets.columns:
            targets[TARGET_COLUMN] = ((101 - targets['Billboard_Rank']) / 100.0).clip(lower=0.0, upper=1.0)
        else:
            targets[TARGET_COLUMN] = pd.to_numeric(targets[TARGET_COLUMN], errors='coerce')
        targets['_chart_key'] = make_song_artist_key(targets, 'SONG_TITLE', 'ARTIST_NAME')
        print(f'Loaded combined Billboard target file: {combined_file}')
        return targets[['SONG_TITLE', 'ARTIST_NAME', 'Billboard_Year', 'Billboard_Rank', TARGET_COLUMN, '_chart_key']]

    rows = []
    chart_files = sorted(Path(chart_dir).glob('Billboard_Top_100_*.csv'))
    if not chart_files:
        print(f'No Billboard chart CSVs found in {chart_dir}.')
        return pd.DataFrame(columns=['SONG_TITLE', 'ARTIST_NAME', 'Billboard_Year', 'Billboard_Rank', TARGET_COLUMN, '_chart_key'])

    for path in chart_files:
        match = re.search(r'(\d{4})', path.stem)
        if not match:
            continue
        year = int(match.group(1))
        chart = pd.read_csv(path)
        if not {'Song Name', 'Artist Name'}.issubset(chart.columns):
            raise ValueError(f'{path} missing Song Name / Artist Name columns')
        chart = chart[['Song Name', 'Artist Name']].copy()
        chart['Billboard_Year'] = year
        chart['Billboard_Rank'] = np.arange(1, len(chart) + 1)
        chart[TARGET_COLUMN] = ((101 - chart['Billboard_Rank']) / 100.0).clip(lower=0.0, upper=1.0)
        chart = chart.rename(columns={'Song Name': 'SONG_TITLE', 'Artist Name': 'ARTIST_NAME'})
        rows.append(chart)

    targets = pd.concat(rows, ignore_index=True)
    targets['_chart_key'] = make_song_artist_key(targets, 'SONG_TITLE', 'ARTIST_NAME')
    return targets

def attach_billboard_targets(df, targets):
    df = df.copy()
    if targets.empty or 'year' not in df.columns:
        return df
    df['_chart_key'] = make_song_artist_key(df, 'SONG_TITLE', 'ARTIST_NAME')
    df['Billboard_Year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    target_lookup = targets[['_chart_key', 'Billboard_Year', 'Billboard_Rank', TARGET_COLUMN]].copy()
    out = df.merge(target_lookup, on=['_chart_key', 'Billboard_Year'], how='left')
    out = out.drop(columns=['_chart_key'])
    return out

missing_identity = [col for col in IDENTITY_COLUMNS if col not in audio_df.columns]
missing_librosa = [col for col in LIBROSA_FEATURE_COLUMNS if col not in audio_df.columns]
if missing_identity:
    raise ValueError(f'Missing identity columns: {missing_identity}')
if missing_librosa:
    raise ValueError(f'Missing expected Librosa columns: {missing_librosa}')

billboard_targets = load_billboard_targets(BILLBOARD_CHART_DIR, BILLBOARD_COMBINED_PATH)
billboard_targets.to_csv(BILLBOARD_TARGETS_PATH, index=False)
audio_df = attach_billboard_targets(audio_df, billboard_targets)

AUDIO_INPUT_COLUMNS = [col for col in OPTIONAL_METADATA_FEATURES + LIBROSA_FEATURE_COLUMNS if col in audio_df.columns]
for col in AUDIO_INPUT_COLUMNS:
    audio_df[col] = pd.to_numeric(audio_df[col], errors='coerce')

print('Rows:', len(audio_df))
print('Columns:', len(audio_df.columns))
print('Audio input columns:', len(AUDIO_INPUT_COLUMNS))
print('Billboard target rows:', len(billboard_targets))
print('Audio rows with HitScore:', int(audio_df[TARGET_COLUMN].notna().sum()) if TARGET_COLUMN in audio_df.columns else 0)
print('Saved target table:', BILLBOARD_TARGETS_PATH)
audio_df[IDENTITY_COLUMNS + AUDIO_INPUT_COLUMNS + [col for col in ['Billboard_Year', 'Billboard_Rank', TARGET_COLUMN] if col in audio_df.columns]].head()

## 4. Clean Feature Matrix

Builds the numeric audio feature matrix and fills missing feature values with column medians.

In [ ]:
model_audio = audio_df[IDENTITY_COLUMNS + AUDIO_INPUT_COLUMNS].copy()
feature_medians = model_audio[AUDIO_INPUT_COLUMNS].median(numeric_only=True)
model_audio[AUDIO_INPUT_COLUMNS] = model_audio[AUDIO_INPUT_COLUMNS].fillna(feature_medians)

print('Rows after cleaning:', len(model_audio))
print('Remaining missing values:', int(model_audio[AUDIO_INPUT_COLUMNS].isna().sum().sum()))

## 5. Optional Audio Hit-Score Model

Runs only if a real target exists. Otherwise, this section prints a skip message and the recommender still runs.

In [ ]:
def attach_target(df):
    df = df.copy()
    if TARGET_COLUMN in df.columns:
        df[TARGET_COLUMN] = pd.to_numeric(df[TARGET_COLUMN], errors='coerce')
        return df, TARGET_COLUMN
    rank_column = next((col for col in RANK_COLUMNS if col in df.columns), None)
    if rank_column is not None:
        rank = pd.to_numeric(df[rank_column], errors='coerce')
        df[TARGET_COLUMN] = ((101 - rank) / 100.0).clip(lower=0, upper=1)
        print(f'Derived {TARGET_COLUMN} from {rank_column}.')
        return df, TARGET_COLUMN
    return df, None

target_audio, target_column = attach_target(audio_df)
hit_score_metrics = None
hit_score_models = {}

if target_column is None:
    print('No hit-score/rank target in master_audio_features.csv. Skipping supervised audio training.')
else:
    supervised = model_audio.merge(target_audio[['SONG_ID', target_column]], on='SONG_ID', how='inner')
    supervised = supervised.dropna(subset=[target_column]).reset_index(drop=True)
    if len(supervised) < 20:
        print(f'Only {len(supervised)} target rows. Skipping until more labels exist.')
    else:
        X = supervised[AUDIO_INPUT_COLUMNS].astype(float)
        y = supervised[target_column].astype(float)
        candidates = {
            'ridge': Pipeline([('scaler', StandardScaler()), ('regressor', Ridge(alpha=10.0))]),
            'random_forest': RandomForestRegressor(n_estimators=500, max_depth=18, min_samples_leaf=2, random_state=SEED, n_jobs=-1),
        }
        rows = []
        for name, model in candidates.items():
            cv = KFold(n_splits=min(5, len(supervised)), shuffle=True, random_state=SEED)
            pred = np.zeros(len(supervised))
            for train_idx, test_idx in cv.split(X):
                model.fit(X.iloc[train_idx], y.iloc[train_idx])
                pred[test_idx] = model.predict(X.iloc[test_idx])
            rows.append({
                'Model': name,
                'Rows': len(supervised),
                'Features': len(AUDIO_INPUT_COLUMNS),
                'Target': target_column,
                'MAE': mean_absolute_error(y, pred),
                'RMSE': mean_squared_error(y, pred) ** 0.5,
                'R2': r2_score(y, pred),
            })
            model.fit(X, y)
            hit_score_models[name] = {'model': model, 'feature_columns': AUDIO_INPUT_COLUMNS, 'target_column': target_column}
        hit_score_metrics = pd.DataFrame(rows).sort_values(['RMSE', 'MAE']).reset_index(drop=True)
        joblib.dump(hit_score_models, AUDIO_HIT_MODELS_PATH)
        print('Saved audio hit-score models:', AUDIO_HIT_MODELS_PATH)
        display(hit_score_metrics.round(4))

## 6. Audio Similarity Index

Scales the audio features and builds a cosine-nearest-neighbor recommender over audio only.

In [ ]:
audio_scaler = StandardScaler()
audio_matrix = audio_scaler.fit_transform(model_audio[AUDIO_INPUT_COLUMNS].astype(float).to_numpy())

recommender = NearestNeighbors(metric='cosine', algorithm='brute')
recommender.fit(audio_matrix)

payload = {
    'model': recommender,
    'vectors': audio_matrix,
    'song_metadata': model_audio[IDENTITY_COLUMNS].reset_index(drop=True),
    'feature_columns': AUDIO_INPUT_COLUMNS,
    'scaler': audio_scaler,
}
joblib.dump(payload, AUDIO_RECOMMENDER_PATH)
print('Saved audio recommender:', AUDIO_RECOMMENDER_PATH)

def recommend_audio_neighbors(row_index, k=10):
    distances, indices = recommender.kneighbors(audio_matrix[[row_index]], n_neighbors=min(k + 1, len(model_audio)))
    rows = []
    for distance, idx in zip(distances[0], indices[0]):
        if idx == row_index:
            continue
        rows.append({
            'SONG_TITLE': model_audio.loc[idx, 'SONG_TITLE'],
            'ARTIST_NAME': model_audio.loc[idx, 'ARTIST_NAME'],
            'Similarity': 1 - distance,
        })
    return pd.DataFrame(rows)

recommend_audio_neighbors(0, k=10)

## 7. Save Metadata

Writes a small JSON summary so the team can see what data and artifacts this run produced.

In [ ]:
metadata = {
    'source_csv': str(MASTER_AUDIO_PATH),
    'rows': int(len(model_audio)),
    'billboard_target_rows': int(len(billboard_targets)),
    'audio_rows_with_hit_score': int(audio_df[TARGET_COLUMN].notna().sum()) if TARGET_COLUMN in audio_df.columns else 0,
    'hit_score_formula': '(101 - Billboard_Rank) / 100',
    'input_columns': AUDIO_INPUT_COLUMNS,
    'input_column_count': len(AUDIO_INPUT_COLUMNS),
    'uses_spotify_api': False,
    'uses_spotify_ids': False,
    'target_column': target_column,
    'hit_score_metrics': hit_score_metrics.round(6).to_dict(orient='records') if hit_score_metrics is not None else None,
    'artifacts': {
        'billboard_targets': str(BILLBOARD_TARGETS_PATH),
        'audio_recommender': str(AUDIO_RECOMMENDER_PATH),
        'audio_hit_models': str(AUDIO_HIT_MODELS_PATH) if hit_score_models else None,
    },
}
AUDIO_METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
metadata